# VGG16 CIFAR10 Training: FP16 Base + FP8 Low Precision

## Mixed-Precision Training dengan GradScaler untuk Underflow Protection

Notebook ini mendemonstrasikan training VGG16 pada CIFAR-10 menggunakan **dual-level mixed precision**:

| Level | Precision | Mekanisme | Diterapkan Pada |
|-------|-----------|-----------|------------------|
| **Base** | FP16 | `torch.cuda.amp.autocast` + `GradScaler` | Semua layer default |
| **Low** | FP8 (E4M3/E5M2) | Native FP8 casting + delayed scaling | Conv2d & Linear terpilih |

### Mengapa GradScaler Diperlukan?

FP16 memiliki dynamic range yang terbatas (~5.96e-8 hingga 65504). Saat training dengan FP16:
- **Underflow**: Gradient kecil bisa menjadi nol (loss of information)
- **Overflow**: Gradient besar bisa menjadi `inf`

`GradScaler` mengatasi ini dengan:
1. **Scaling loss** sebelum backward pass (mencegah underflow)
2. **Unscaling gradients** sebelum optimizer step
3. **Skipping steps** jika ditemukan `inf`/`nan` gradients
4. **Dynamic scale adjustment** — naik jika stabil, turun jika overflow

### FP8 Layer Selection Strategy

- **E4M3** (4-bit exponent, 3-bit mantissa): Digunakan untuk forward pass — range ±448
- **E5M2** (5-bit exponent, 2-bit mantissa): Digunakan untuk backward pass — range ±57344

Layer-layer awal (conv) dan layer pertama FC dipilih untuk FP8 karena
memiliki toleransi lebih tinggi terhadap quantization noise.

In [ ]:
# ============================================================
# Cell 2: Imports & Environment Check
# ============================================================
import sys
import os
import time
import math

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.cuda.amp import GradScaler, autocast

# ---- Tambahkan project root ke sys.path ----
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'[INFO] Project root: {PROJECT_ROOT}')

# ---- Import ext3 modules ----
from ext3.nn.nn_native import (
    NativeConv2d, NativeLinear, NativeBatchNorm2d,
    NativeReLU, NativeMaxPool2d, NativeAdaptiveAvgPool2d,
    NativeDropout,
)
from ext3.core.include.native_precision import (
    NativePrecisionMode, FP8ScalingManager, FP8Config,
    check_fp8_support, enable_tf32, disable_tf32,
)

# ---- Environment Check ----
print('=' * 60)
print('ENVIRONMENT CHECK')
print('=' * 60)
print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name         : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory       : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
    cap = torch.cuda.get_device_capability(0)
    print(f'Compute cap      : {cap[0]}.{cap[1]}')
print(f'FP8 support      : {check_fp8_support()}')
print(f'Torchvision ver  : {torchvision.__version__}')
print('=' * 60)

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================
CONFIG = {
    'batch_size': 128,
    'epochs': 50,
    'lr': 0.01,
    'momentum': 0.9,
    'weight_decay': 5e-4,
    'num_workers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    # FP8 assignment — index dari conv/linear layers yang akan menggunakan FP8
    'fp8_conv_layers': [0, 1, 2, 3, 4, 5, 6],
    'fp8_linear_layers': [0],
    # GradScaler config — initial scale dan growth interval
    'grad_scaler_init_scale': 2.**16,
    'grad_scaler_growth_interval': 2000,
}

# NOTE: TF32 NOT enabled here — kita menggunakan FP16 sebagai base precision
# TF32 akan mengurangi efek FP16 autocast karena matmul sudah reduced precision
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

print('[CONFIG] Training Configuration:')
for k, v in CONFIG.items():
    print(f'  {k:30s}: {v}')
print(f'\n[CONFIG] TF32 matmul  : {torch.backends.cuda.matmul.allow_tf32}')
print(f'[CONFIG] TF32 cuDNN  : {torch.backends.cudnn.allow_tf32}')
print(f'[CONFIG] Base mode   : FP16 (autocast + GradScaler)')
print(f'[CONFIG] Low mode    : FP8 E4M3 (forward) / E5M2 (backward)')

In [ ]:
# ============================================================
# Cell 4: VGG16 Model Definition
# ============================================================
# Arsitektur VGG16 dengan Native Precision layers dari ext3
# Menggunakan NativeConv2d, NativeLinear, dll. agar tiap layer
# bisa di-assign precision mode secara independen.

class VGG16Native(nn.Module):
    """
    VGG16 menggunakan Native Precision layers.
    
    Arsitektur:
    - 13 Conv2d layers (NativeConv2d) dalam 5 blok
    - 3 Linear layers (NativeLinear) sebagai classifier
    - BatchNorm2d setelah setiap Conv2d (modern VGG variant)
    - Disesuaikan untuk CIFAR-10 (input 32x32, 10 classes)
    """
    
    def __init__(self, num_classes=10):
        super(VGG16Native, self).__init__()
        
        # ---- Feature Extractor (Convolutional Blocks) ----
        self.features = nn.Sequential(
            # Block 1: 3 -> 64, 2 conv layers
            NativeConv2d(3, 64, kernel_size=3, padding=1),       # conv_layers[0]
            NativeBatchNorm2d(64),
            NativeReLU(inplace=True),
            NativeConv2d(64, 64, kernel_size=3, padding=1),      # conv_layers[1]
            NativeBatchNorm2d(64),
            NativeReLU(inplace=True),
            NativeMaxPool2d(kernel_size=2, stride=2),
            
            # Block 2: 64 -> 128, 2 conv layers
            NativeConv2d(64, 128, kernel_size=3, padding=1),     # conv_layers[2]
            NativeBatchNorm2d(128),
            NativeReLU(inplace=True),
            NativeConv2d(128, 128, kernel_size=3, padding=1),    # conv_layers[3]
            NativeBatchNorm2d(128),
            NativeReLU(inplace=True),
            NativeMaxPool2d(kernel_size=2, stride=2),
            
            # Block 3: 128 -> 256, 3 conv layers
            NativeConv2d(128, 256, kernel_size=3, padding=1),    # conv_layers[4]
            NativeBatchNorm2d(256),
            NativeReLU(inplace=True),
            NativeConv2d(256, 256, kernel_size=3, padding=1),    # conv_layers[5]
            NativeBatchNorm2d(256),
            NativeReLU(inplace=True),
            NativeConv2d(256, 256, kernel_size=3, padding=1),    # conv_layers[6]
            NativeBatchNorm2d(256),
            NativeReLU(inplace=True),
            NativeMaxPool2d(kernel_size=2, stride=2),
            
            # Block 4: 256 -> 512, 3 conv layers
            NativeConv2d(256, 512, kernel_size=3, padding=1),    # conv_layers[7]
            NativeBatchNorm2d(512),
            NativeReLU(inplace=True),
            NativeConv2d(512, 512, kernel_size=3, padding=1),    # conv_layers[8]
            NativeBatchNorm2d(512),
            NativeReLU(inplace=True),
            NativeConv2d(512, 512, kernel_size=3, padding=1),    # conv_layers[9]
            NativeBatchNorm2d(512),
            NativeReLU(inplace=True),
            NativeMaxPool2d(kernel_size=2, stride=2),
            
            # Block 5: 512 -> 512, 3 conv layers
            NativeConv2d(512, 512, kernel_size=3, padding=1),    # conv_layers[10]
            NativeBatchNorm2d(512),
            NativeReLU(inplace=True),
            NativeConv2d(512, 512, kernel_size=3, padding=1),    # conv_layers[11]
            NativeBatchNorm2d(512),
            NativeReLU(inplace=True),
            NativeConv2d(512, 512, kernel_size=3, padding=1),    # conv_layers[12]
            NativeBatchNorm2d(512),
            NativeReLU(inplace=True),
            NativeMaxPool2d(kernel_size=2, stride=2),
        )
        
        # ---- Adaptive Pooling ----
        self.avgpool = NativeAdaptiveAvgPool2d((1, 1))
        
        # ---- Classifier (Fully Connected Layers) ----
        self.classifier = nn.Sequential(
            NativeLinear(512, 512),                               # linear_layers[0]
            NativeReLU(inplace=True),
            NativeDropout(0.5),
            NativeLinear(512, 512),                               # linear_layers[1]
            NativeReLU(inplace=True),
            NativeDropout(0.5),
            NativeLinear(512, num_classes),                       # linear_layers[2]
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# Quick test
print('[INFO] VGG16Native architecture created.')
_test_model = VGG16Native(num_classes=10)
_n_params = sum(p.numel() for p in _test_model.parameters())
print(f'[INFO] Total parameters: {_n_params:,}')

# Count layer types
_n_conv = sum(1 for m in _test_model.modules() if isinstance(m, NativeConv2d))
_n_linear = sum(1 for m in _test_model.modules() if isinstance(m, NativeLinear))
print(f'[INFO] NativeConv2d layers  : {_n_conv}')
print(f'[INFO] NativeLinear layers  : {_n_linear}')
del _test_model

In [ ]:
# ============================================================
# Cell 5: Precision Assignment Function
# ============================================================
# Mengassign precision mode ke setiap layer:
# - Layer terpilih → 'low_fp8' (FP8 native casting + delayed scaling)
# - Layer lainnya  → 'base_fp16' (FP16 autocast per-layer)

def assign_precision(model, config):
    """
    Assign native precision mode ke setiap NativeConv2d dan NativeLinear.
    
    Layer yang ada di fp8_conv_layers / fp8_linear_layers → 'low_fp8'
    Layer sisanya → 'base_fp16'
    
    NOTE: Berbeda dengan TF32 notebook, di sini base mode adalah FP16,
    bukan TF32. Ini karena kita menggunakan autocast + GradScaler.
    
    Args:
        model: VGG16Native model instance
        config: Dictionary konfigurasi
    """
    fp8_conv_indices = set(config.get('fp8_conv_layers', []))
    fp8_linear_indices = set(config.get('fp8_linear_layers', []))
    
    # ---- Assign Conv2d layers ----
    conv_idx = 0
    for module in model.modules():
        if isinstance(module, NativeConv2d):
            if conv_idx in fp8_conv_indices:
                module.set_native_precision('low_fp8')
                mode_str = 'low_fp8'
            else:
                module.set_native_precision('base_fp16')
                mode_str = 'base_fp16'
            print(f'  Conv2d[{conv_idx:2d}] -> {mode_str:12s}  '
                  f'({module.in_channels}->{module.out_channels}, '
                  f'k={module.kernel_size})')
            conv_idx += 1
    
    # ---- Assign Linear layers ----
    linear_idx = 0
    for module in model.modules():
        if isinstance(module, NativeLinear):
            if linear_idx in fp8_linear_indices:
                module.set_native_precision('low_fp8')
                mode_str = 'low_fp8'
            else:
                module.set_native_precision('base_fp16')
                mode_str = 'base_fp16'
            print(f'  Linear[{linear_idx:2d}] -> {mode_str:12s}  '
                  f'({module.in_features}->{module.out_features})')
            linear_idx += 1
    
    # ---- Summary ----
    n_fp8 = sum(
        1 for m in model.modules()
        if hasattr(m, '_native_mode') and m._native_mode == NativePrecisionMode.LOW_FP8
    )
    n_fp16 = sum(
        1 for m in model.modules()
        if hasattr(m, '_native_mode') and m._native_mode == NativePrecisionMode.BASE_FP16
    )
    print(f'\n[SUMMARY] FP8 layers  : {n_fp8}')
    print(f'[SUMMARY] FP16 layers : {n_fp16}')
    print(f'[SUMMARY] Total assigned: {n_fp8 + n_fp16}')


# Test assignment
print('[INFO] Precision Assignment Preview:')
print('=' * 60)
_test_model = VGG16Native(num_classes=10)
assign_precision(_test_model, CONFIG)
del _test_model
print('=' * 60)

In [ ]:
# ============================================================
# Cell 6: CIFAR-10 Data Pipeline
# ============================================================
# Standard data augmentation untuk CIFAR-10:
# - RandomCrop 32x32 dengan padding 4
# - RandomHorizontalFlip
# - Normalize dengan mean/std CIFAR-10

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2023, 0.1994, 0.2010),
    ),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2023, 0.1994, 0.2010),
    ),
])

# Download dan load dataset
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=CONFIG['num_workers'],
    pin_memory=True,
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=CONFIG['num_workers'],
    pin_memory=True,
)

print(f'[DATA] Training samples  : {len(trainset):,}')
print(f'[DATA] Test samples      : {len(testset):,}')
print(f'[DATA] Training batches  : {len(trainloader)}')
print(f'[DATA] Test batches      : {len(testloader)}')
print(f'[DATA] Batch size        : {CONFIG["batch_size"]}')
print(f'[DATA] Classes           : {trainset.classes}')

In [ ]:
# ============================================================
# Cell 7: VRAM Monitoring + Loss Stability Checker
# ============================================================

def get_vram_mb():
    """Return current GPU VRAM usage in MB."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024**2)
    return 0.0


class LossStabilityChecker:
    """
    Detects NaN/Inf losses and triggers FP8 -> FP16 fallback.
    
    Ketika FP8 quantization terlalu agresif, loss bisa menjadi NaN/Inf.
    Checker ini menghitung berapa kali berturut-turut loss abnormal.
    Jika melebihi `patience`, trigger fallback ke FP16.
    
    Attributes:
        nan_count: Consecutive NaN/Inf count
        patience: Maximum consecutive NaN/Inf sebelum fallback
        triggered: True jika fallback sudah di-trigger
    """
    
    def __init__(self, patience=3):
        self.nan_count = 0
        self.patience = patience
        self.triggered = False
    
    def check(self, loss_val):
        """
        Check apakah loss value abnormal.
        
        Args:
            loss_val: Float loss value dari training
            
        Returns:
            bool: True jika harus disable FP8 (fallback triggered)
        """
        if math.isnan(loss_val) or math.isinf(loss_val):
            self.nan_count += 1
            print(f'  [STABILITY] NaN/Inf detected! Count: {self.nan_count}/{self.patience}')
            if self.nan_count >= self.patience:
                self.triggered = True
                return True  # Signal to disable FP8
        else:
            self.nan_count = 0  # Reset counter on normal loss
        return False


print('[INFO] VRAM monitor and LossStabilityChecker ready.')
print(f'[INFO] Current VRAM usage: {get_vram_mb():.1f} MB')

In [ ]:
# ============================================================
# Cell 8: Training Function with GradScaler
# ============================================================
# Training loop menggunakan:
# - torch.cuda.amp.autocast(dtype=torch.float16) untuk forward pass
# - GradScaler untuk backward pass (loss scaling)
# - Gradient clipping (max_norm=1.0) untuk stabilitas
# - Overflow detection via scale factor monitoring

def train_epoch(model, loader, criterion, optimizer, scaler, device, stability_checker):
    """
    Train model untuk satu epoch dengan FP16 autocast + GradScaler.
    
    Pipeline per batch:
    1. Forward pass dalam autocast FP16 context
       - Layer FP8 melakukan cast internal ke FP8 (dalam autocast context)
       - Layer FP16 menggunakan autocast biasa
    2. GradScaler.scale(loss) → scaled backward pass
    3. GradScaler.unscale_(optimizer) → unscale gradients
    4. Gradient clipping (max_norm=1.0)
    5. GradScaler.step(optimizer) → skip jika inf/nan grads
    6. GradScaler.update() → adjust scale factor
    
    Args:
        model: VGG16Native model
        loader: Training DataLoader
        criterion: Loss function
        optimizer: Optimizer
        scaler: GradScaler instance
        device: CUDA device
        stability_checker: LossStabilityChecker instance
    
    Returns:
        tuple: (avg_loss, accuracy%, overflow_count)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    overflow_count = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        
        # ---- Forward with autocast FP16 ----
        # Semua operasi dalam context ini akan menggunakan FP16 secara default.
        # Layer dengan mode 'low_fp8' akan melakukan FP8 casting tambahan
        # di dalam forward() mereka (via NativePrecisionMixin).
        with torch.cuda.amp.autocast(dtype=torch.float16):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        
        # ---- Backward with GradScaler ----
        # GradScaler memperbesar loss sebelum backward untuk mencegah
        # gradient underflow di FP16
        scaler.scale(loss).backward()
        
        # ---- Unscale gradients ----
        # Kembalikan gradient ke skala asli sebelum clipping
        scaler.unscale_(optimizer)
        
        # ---- Gradient clipping ----
        # Mencegah gradient explosion, terutama penting dengan FP8
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # ---- Optimizer step ----
        # GradScaler.step() akan SKIP optimizer.step() jika
        # ada inf/nan gradients (detected during unscale_)
        scaler.step(optimizer)
        
        # ---- Update scale factor ----
        scale_before = scaler.get_scale()
        scaler.update()
        scale_after = scaler.get_scale()
        
        # Detect overflow: jika scale turun, berarti ada overflow
        if scale_after < scale_before:
            overflow_count += 1
        
        # ---- Accumulate metrics ----
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    
    # Check stability — apakah loss NaN/Inf?
    stability_checker.check(avg_loss)
    
    return avg_loss, accuracy, overflow_count


print('[INFO] train_epoch() function defined.')

In [ ]:
# ============================================================
# Cell 9: Evaluation Function
# ============================================================
# Evaluasi juga menggunakan autocast FP16 agar konsisten
# dengan kondisi training (precision yang sama).

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Evaluate model pada test set dengan autocast FP16.
    
    Args:
        model: VGG16Native model (eval mode)
        loader: Test DataLoader
        criterion: Loss function
        device: CUDA device
    
    Returns:
        tuple: (avg_loss, accuracy%)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Eval juga dengan autocast FP16
        with torch.cuda.amp.autocast(dtype=torch.float16):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


print('[INFO] evaluate() function defined.')

In [ ]:
# ============================================================
# Cell 10: FP8 Fallback Function
# ============================================================
# Emergency fallback: jika training tidak stabil (NaN/Inf berulang),
# semua FP8 layers di-switch ke FP16 untuk menyelamatkan training.

def disable_fp8_layers(model):
    """
    Emergency fallback: switch all FP8 layers to FP16.
    
    Dipanggil ketika LossStabilityChecker mendeteksi terlalu banyak
    NaN/Inf losses berturut-turut. Ini menunjukkan bahwa FP8 quantization
    terlalu agresif untuk model/data ini.
    
    Setelah fallback, training berlanjut sepenuhnya dalam FP16.
    
    Args:
        model: VGG16Native model instance
    """
    fallback_count = 0
    for module in model.modules():
        if hasattr(module, '_native_mode') and module._native_mode == NativePrecisionMode.LOW_FP8:
            module.set_native_precision('base_fp16')
            fallback_count += 1
    
    print(f'\n{"=" * 60}')
    print(f'[WARNING] FP8 layers disabled — falling back to FP16')
    print(f'[WARNING] due to training instability (NaN/Inf detected).')
    print(f'[WARNING] {fallback_count} layers switched from FP8 -> FP16.')
    print(f'{"=" * 60}\n')


print('[INFO] disable_fp8_layers() fallback function defined.')

In [ ]:
# ============================================================
# Cell 11: Main Training Loop
# ============================================================

# ---- Initialize Model ----
device = CONFIG['device']
model = VGG16Native(num_classes=10).to(device)

# ---- Assign Precision Modes ----
print('[INIT] Assigning precision modes...')
assign_precision(model, CONFIG)

# ---- Optimizer & Criterion ----
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=CONFIG['lr'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
)

# ---- Learning Rate Scheduler ----
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs']
)

# ---- GradScaler ----
# GradScaler mengelola dynamic loss scaling untuk FP16 training.
# init_scale: Skala awal (2^16 = 65536)
# growth_interval: Berapa steps sebelum mencoba menaikkan scale
scaler = GradScaler(
    init_scale=CONFIG['grad_scaler_init_scale'],
    growth_interval=CONFIG['grad_scaler_growth_interval'],
)

# ---- Stability Checker ----
stability_checker = LossStabilityChecker(patience=3)

# ---- Tracking Metrics ----
train_losses = []
train_accs = []
test_losses = []
test_accs = []
vram_usage = []
overflow_counts = []
scaler_scales = []
epoch_times = []
best_test_acc = 0.0
fp8_disabled_epoch = None  # Track kapan FP8 di-disable (jika terjadi)

print(f'\n{"=" * 70}')
print(f'{"TRAINING START":^70}')
print(f'  Mode: FP16 Base + FP8 Low Precision with GradScaler')
print(f'  Epochs: {CONFIG["epochs"]}, LR: {CONFIG["lr"]}, Batch: {CONFIG["batch_size"]}')
print(f'  GradScaler init scale: {CONFIG["grad_scaler_init_scale"]:.0f}')
print(f'{"=" * 70}\n')

total_start_time = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):
    epoch_start = time.time()
    
    # ---- Check stability: disable FP8 if triggered ----
    if stability_checker.triggered and fp8_disabled_epoch is None:
        disable_fp8_layers(model)
        fp8_disabled_epoch = epoch
    
    # ---- Train ----
    train_loss, train_acc, n_overflow = train_epoch(
        model, trainloader, criterion, optimizer, scaler, device, stability_checker
    )
    
    # ---- Evaluate ----
    test_loss, test_acc = evaluate(model, testloader, criterion, device)
    
    # ---- Step scheduler ----
    scheduler.step()
    
    # ---- Record metrics ----
    epoch_time = time.time() - epoch_start
    current_vram = get_vram_mb()
    current_scale = scaler.get_scale()
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    vram_usage.append(current_vram)
    overflow_counts.append(n_overflow)
    scaler_scales.append(current_scale)
    epoch_times.append(epoch_time)
    
    # ---- Track best ----
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_marker = ' *** BEST ***'
    else:
        best_marker = ''
    
    # ---- Print epoch summary ----
    print(
        f'Epoch {epoch:3d}/{CONFIG["epochs"]} | '
        f'Train Loss: {train_loss:.4f} | '
        f'Train Acc: {train_acc:6.2f}% | '
        f'Test Acc: {test_acc:6.2f}% | '
        f'VRAM: {current_vram:7.1f}MB | '
        f'Overflow: {n_overflow:3d} | '
        f'Scale: {current_scale:.0f} | '
        f'Time: {epoch_time:.1f}s'
        f'{best_marker}'
    )

total_time = time.time() - total_start_time

print(f'\n{"=" * 70}')
print(f'{"TRAINING COMPLETE":^70}')
print(f'{"=" * 70}')
print(f'  Total time       : {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'  Best test acc    : {best_test_acc:.2f}%')
print(f'  Final train loss : {train_losses[-1]:.4f}')
print(f'  Final test acc   : {test_accs[-1]:.2f}%')
print(f'  Final scale      : {scaler_scales[-1]:.0f}')
print(f'  Total overflows  : {sum(overflow_counts)}')
print(f'  Avg epoch time   : {sum(epoch_times)/len(epoch_times):.1f}s')
if fp8_disabled_epoch is not None:
    print(f'  FP8 disabled at  : epoch {fp8_disabled_epoch}')
else:
    print(f'  FP8 status       : Active throughout training')
print(f'{"=" * 70}')

In [ ]:
# ============================================================
# Cell 12: Visualization
# ============================================================
# 4 subplots menampilkan metrik training:
# 1. Loss curve (train & test)
# 2. Accuracy curve (train & test)
# 3. VRAM usage per epoch
# 4. Gradient Scale & Overflow count

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'VGG16 CIFAR-10: FP16 Base + FP8 Low Precision Training',
    fontsize=16, fontweight='bold', y=0.98
)
epochs_range = range(1, len(train_losses) + 1)

# ---- Subplot 1: Loss Curve ----
ax1 = axes[0, 0]
ax1.plot(epochs_range, train_losses, 'b-', linewidth=1.5, label='Train Loss', alpha=0.8)
ax1.plot(epochs_range, test_losses, 'r--', linewidth=1.5, label='Test Loss', alpha=0.8)
if fp8_disabled_epoch is not None:
    ax1.axvline(x=fp8_disabled_epoch, color='orange', linestyle=':', linewidth=2,
                label=f'FP8 Disabled (epoch {fp8_disabled_epoch})')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training & Test Loss', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, len(train_losses))

# ---- Subplot 2: Accuracy Curve ----
ax2 = axes[0, 1]
ax2.plot(epochs_range, train_accs, 'b-', linewidth=1.5, label='Train Acc', alpha=0.8)
ax2.plot(epochs_range, test_accs, 'r--', linewidth=1.5, label='Test Acc', alpha=0.8)
ax2.axhline(y=best_test_acc, color='green', linestyle=':', linewidth=1,
            label=f'Best Test: {best_test_acc:.2f}%', alpha=0.7)
if fp8_disabled_epoch is not None:
    ax2.axvline(x=fp8_disabled_epoch, color='orange', linestyle=':', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training & Test Accuracy', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='lower right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1, len(train_losses))

# ---- Subplot 3: VRAM Usage ----
ax3 = axes[1, 0]
ax3.fill_between(epochs_range, vram_usage, alpha=0.3, color='purple')
ax3.plot(epochs_range, vram_usage, 'purple', linewidth=1.5, label='VRAM (MB)')
if fp8_disabled_epoch is not None:
    ax3.axvline(x=fp8_disabled_epoch, color='orange', linestyle=':', linewidth=2,
                label=f'FP8 Disabled')
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('VRAM (MB)', fontsize=12)
ax3.set_title('GPU VRAM Usage', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(1, len(train_losses))

# ---- Subplot 4: Gradient Scale & Overflow Count ----
ax4 = axes[1, 1]

# Plot scaler scale pada axis kiri (log scale)
color_scale = 'tab:blue'
ax4.set_xlabel('Epoch', fontsize=12)
ax4.set_ylabel('GradScaler Scale', fontsize=12, color=color_scale)
ax4.plot(epochs_range, scaler_scales, color=color_scale, linewidth=1.5,
         label='GradScaler Scale', alpha=0.8)
ax4.tick_params(axis='y', labelcolor=color_scale)
ax4.set_yscale('log')

# Plot overflow count pada axis kanan
ax4_twin = ax4.twinx()
color_overflow = 'tab:red'
ax4_twin.set_ylabel('Overflow Count', fontsize=12, color=color_overflow)
ax4_twin.bar(list(epochs_range), overflow_counts, alpha=0.4, color=color_overflow,
             label='Overflow Count')
ax4_twin.tick_params(axis='y', labelcolor=color_overflow)

if fp8_disabled_epoch is not None:
    ax4.axvline(x=fp8_disabled_epoch, color='orange', linestyle=':', linewidth=2)

ax4.set_title('GradScaler Scale & Overflow Events', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.set_xlim(1, len(train_losses))

# Combine legends
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4_twin.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper right')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('vgg16_fp16_fp8_training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n[INFO] Plot saved to: vgg16_fp16_fp8_training_results.png')

# ---- Summary Statistics ----
print(f'\n{"=" * 50}')
print(f'{"TRAINING SUMMARY":^50}')
print(f'{"=" * 50}')
print(f'  Final Train Loss  : {train_losses[-1]:.4f}')
print(f'  Final Train Acc   : {train_accs[-1]:.2f}%')
print(f'  Final Test Loss   : {test_losses[-1]:.4f}')
print(f'  Final Test Acc    : {test_accs[-1]:.2f}%')
print(f'  Best Test Acc     : {best_test_acc:.2f}%')
print(f'  Peak VRAM         : {max(vram_usage):.1f} MB')
print(f'  Total Overflows   : {sum(overflow_counts)}')
print(f'  Final Scale       : {scaler_scales[-1]:.0f}')
print(f'{"=" * 50}')

## Analisis: FP8 Scaling Mechanism & GradScaler Interaction

### 1. Mekanisme FP8 Delayed Scaling

FP8 menggunakan **delayed scaling** untuk menentukan scale factor yang optimal:

```
Iterasi t:
  1. Ambil scale factor dari history (dihitung di iterasi t-1)
  2. Cast input/weight ke FP8: tensor_fp8 = (tensor * scale).clamp(-max_val, max_val).to(fp8)
  3. Execute forward pass
  4. Record amax (max absolute value) dari tensor asli
  5. Update scale: scale = dtype_max / max(amax_history)
```

Scale factor "delayed by 1 step" karena dihitung berdasarkan statistics dari iterasi sebelumnya.
Ini lebih stabil daripada per-tensor scaling yang langsung menggunakan amax dari batch saat ini.

### 2. Interaksi GradScaler dengan FP8

Dalam notebook ini, ada **dua level scaling** yang bekerja bersamaan:

| Level | Mekanisme | Tujuan |
|-------|-----------|--------|
| **GradScaler** (training loop) | Dynamic loss scaling | Mencegah gradient underflow di FP16 |
| **FP8ScalingManager** (per-layer) | Delayed per-tensor scaling | Memetakan tensor ke FP8 range |

Keduanya bekerja **independen** dan **komplementer**:
- GradScaler mengurus **backward pass** (gradient magnitudes)
- FP8ScalingManager mengurus **forward pass** (activation & weight magnitudes)

### 3. Pencegahan Underflow & Overflow

#### FP16 Underflow Prevention (via GradScaler):
- **Problem**: FP16 min subnormal = 5.96e-8. Gradien kecil bisa menjadi 0.
- **Solution**: Scale loss sebelum backward → gradien diperbesar → tidak underflow.
- **Dynamic adjustment**: Jika ada overflow (inf grads), scale diturunkan 2x. Jika stabil selama `growth_interval` steps, scale dinaikkan 2x.

#### FP8 Range Management (via FP8ScalingManager):
- **Problem**: FP8 E4M3 max = 448, E5M2 max = 57344. Range sangat terbatas.
- **Solution**: Per-tensor scale factor memetakan dynamic range tensor ke FP8 representable range.
- **Delayed scaling**: Menggunakan history `amax` dari 16 iterasi terakhir untuk stabilitas.

### 4. Mengapa E4M3 untuk Forward dan E5M2 untuk Backward?

| Format | Exponent | Mantissa | Range | Precision | Digunakan Untuk |
|--------|----------|----------|-------|-----------|------------------|
| **E4M3** | 4 bit | 3 bit | ±448 | Lebih tinggi | Forward (weight & activation) |
| **E5M2** | 5 bit | 2 bit | ±57344 | Lebih rendah | Backward (gradients) |

**Alasan pemilihan:**

1. **Forward pass (E4M3)**:
   - Weights dan activations biasanya memiliki distribusi yang **compact** (range kecil)
   - Precision lebih penting daripada range → E4M3 (3-bit mantissa) memberikan resolusi lebih baik
   - Dengan delayed scaling, range ±448 sudah cukup

2. **Backward pass (E5M2)**:
   - Gradients memiliki distribusi **wide** dan bisa sangat kecil atau sangat besar
   - Dynamic range lebih penting → E5M2 (5-bit exponent) memberikan range ±57344
   - Precision lebih rendah bisa ditoleransi karena gradients di-aggregate (averaged)

### 5. Overflow Detection & Fallback

Notebook ini memiliki mekanisme **safety net** berlapis:

1. **GradScaler overflow detection**: Skip optimizer step jika ada inf gradients
2. **Gradient clipping** (max_norm=1.0): Mencegah gradient explosion
3. **LossStabilityChecker**: Monitor NaN/Inf loss, trigger FP8→FP16 fallback jika berulang
4. **FP8 hardware fallback**: Jika GPU tidak support FP8, otomatis fallback ke FP16

Mekanisme berlapis ini memastikan training tetap konvergen meskipun menggunakan precision sangat rendah.